# 01 · Data Exploration — understanding the raw tables

**Goal:** look at how the data sits in `inventory.db` and decide whether we need an
aggregated table that helps with:
1. Vendor selection for profitability
2. Product pricing optimization

Run cell-by-cell in VS Code (click **Run Cell** above each `# %%`) or the whole file with
`python analysis/01_data_exploration.py`. Requires `inventory.db` (see `data/README.md`).

In [1]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# works from the repo root, from analysis/, or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT / "src"))
from get_vendor_summary import clean_data, create_vendor_summary  # noqa: E402

conn = sqlite3.connect(ROOT / "inventory.db")

## Which tables are in the database and how big are they?

In [2]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name != 'vendor_sales_summary'", conn
)
for table in tables["name"]:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", conn)["n"].iloc[0]
    print("-" * 40, table, f"({count:,} rows)", "-" * 40)
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", conn))

---------------------------------------- begin_inventory (206,529 rows) ----------------------------------------


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,8,12.99,2024-01-01
1,1_HARDERSFIELD_60,1,HARDERSFIELD,60,Canadian Club 1858 VAP,750mL,7,10.99,2024-01-01
2,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,6,36.99,2024-01-01
3,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,3,38.99,2024-01-01
4,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,6,34.99,2024-01-01


---------------------------------------- end_inventory (224,489 rows) ----------------------------------------


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,endDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,11,12.99,2024-12-31
1,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,7,36.99,2024-12-31
2,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,7,38.99,2024-12-31
3,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,4,34.99,2024-12-31
4,1_HARDERSFIELD_75,1,HARDERSFIELD,75,Three Olives Tomato Vodka,750mL,7,14.99,2024-12-31


---------------------------------------- purchases (2,372,474 rows) ----------------------------------------


,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


... (2 outputs left out)

---------------------------------------- sales (12,825,363 rows) ----------------------------------------


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-01,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,2024-01-02,750.0,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-03,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
3,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,14.49,14.49,2024-01-08,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
4,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,2,69.98,34.99,2024-01-09,375.0,1,0.79,12546,JIM BEAM BRANDS COMPANY


---------------------------------------- vendor_invoice (5,543 rows) ----------------------------------------


,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None


## Drill into one vendor (4466 — American Vintage Beverage) to see how tables relate

In [3]:
purchases = pd.read_sql_query("SELECT * FROM purchases WHERE VendorNumber = 4466", conn)
purchase_prices = pd.read_sql_query("SELECT * FROM purchase_prices WHERE VendorNumber = 4466", conn)
vendor_invoice = pd.read_sql_query("SELECT * FROM vendor_invoice WHERE VendorNumber = 4466", conn)
sales = pd.read_sql_query("SELECT * FROM sales WHERE VendorNo = 4466", conn)

print("purchases:", purchases.shape, "| purchase_prices:", purchase_prices.shape,
      "| vendor_invoice:", vendor_invoice.shape, "| sales:", sales.shape)

purchases: (2192, 16) | purchase_prices: (3, 9) | vendor_invoice: (55, 10) | sales: (9453, 14)


In [4]:
display(purchases.groupby(["Brand", "PurchasePrice"])[["Quantity", "Dollars"]].sum())

,,Quantity,Dollars
Brand,PurchasePrice,,
3140,11.19,4640,51921.60
5215,9.41,4923,46325.43
5255,9.35,6215,58110.25


In [5]:
display(sales.groupby("Brand")[["SalesDollars", "SalesPrice", "SalesQuantity"]].sum())

,SalesDollars,SalesPrice,SalesQuantity
Brand,,,
3140,50531.10,30071.85,3890
5215,60416.49,41542.02,4651
5255,79187.04,51180.60,6096


### What we learned
1. **purchases** — actual purchase lines: PO date, receiving date, brand bought from each vendor,
   dollars paid and quantity.
2. **purchase_prices** — the price list: selling price and purchase price per product
   (one row per brand).
3. **vendor_invoice** — one row per vendor per PO; it summarises purchases (quantity, dollars)
   and adds **freight**.
4. **sales** — every sale: brand, quantity sold, selling price, revenue, excise tax.

The numbers we need are spread across four tables, so we build **one summary table** with:
purchases by vendor, sales, freight per vendor and actual product prices.

## Building blocks of the summary query

In [6]:
freight_summary = pd.read_sql_query(
    """SELECT VendorNumber, SUM(Freight) AS FreightCost
       FROM vendor_invoice
       GROUP BY VendorNumber""",
    conn,
)
display(freight_summary.head())

,VendorNumber,FreightCost
0,2,27.08
1,54,0.48
2,60,367.52
3,105,62.39
4,200,6.19


In [7]:
purchase_summary = pd.read_sql_query(
    """SELECT p.VendorNumber, p.VendorName, p.Brand, p.PurchasePrice,
              pp.Volume, pp.Price AS ActualPrice,
              SUM(p.Quantity) AS TotalPurchaseQuantity,
              SUM(p.Dollars)  AS TotalPurchaseDollars
       FROM purchases p
       JOIN purchase_prices pp ON p.Brand = pp.Brand
       WHERE p.PurchasePrice > 0
       GROUP BY p.VendorNumber, p.VendorName, p.Brand
       ORDER BY TotalPurchaseDollars DESC""",
    conn,
)
display(purchase_summary.head())

,VendorNumber,VendorName,Brand,PurchasePrice,Volume,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars
0,1128,BROWN-FORMAN CORP,1233,26.27,1750,36.99,145080,3811251.60
1,4425,MARTIGNETTI COMPANIES,3405,23.19,1750,28.99,164038,3804041.22
2,17035,PERNOD RICARD USA,8068,18.24,1750,24.99,187407,3418303.68
3,3960,DIAGEO NORTH AMERICA INC,4261,16.17,1750,22.99,201682,3261197.94
4,3960,DIAGEO NORTH AMERICA INC,3545,21.89,1750,29.99,138109,3023206.01


In [8]:
sales_summary = pd.read_sql_query(
    """SELECT VendorNo, Brand,
              SUM(SalesDollars)  AS TotalSalesDollars,
              SUM(SalesPrice)    AS TotalSalesPrice,
              SUM(SalesQuantity) AS TotalSalesQuantity,
              SUM(ExciseTax)     AS TotalExciseTax
       FROM sales
       GROUP BY VendorNo, Brand
       ORDER BY TotalSalesDollars DESC""",
    conn,
)
display(sales_summary.head())

,VendorNo,Brand,TotalSalesDollars,TotalSalesPrice,TotalSalesQuantity,TotalExciseTax
0,1128,1233,5.101920e+06,672819.31,142049,260999.20
1,4425,3405,4.819073e+06,561512.37,160247,294438.66
2,17035,8068,4.538121e+06,461140.15,187140,343854.07
3,3960,4261,4.475973e+06,420050.01,200412,368242.80
4,3960,3545,4.223108e+06,545778.28,135838,249587.83


## The full summary query
The three pieces above are combined with CTEs in `sql/vendor_sales_summary.sql`.

**Freight fix:** freight is only known per vendor. The first version copied each vendor's
total freight onto every one of its products, so summing freight over-counted it ~400×.
The query now splits a vendor's freight across its products by share of purchase dollars.

In [9]:
vendor_sales_summary = create_vendor_summary(conn)
print(vendor_sales_summary.shape)
display(vendor_sales_summary.head())

(10692, 14)


,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750,145080,3811251.60,142049.0,5.101920e+06,672819.31,260999.20,19325.145490
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750,164038,3804041.22,160247.0,4.819073e+06,561512.37,294438.66,19787.629629
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750,187407,3418303.68,187140.0,4.538121e+06,461140.15,343854.07,17539.246213
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750,201682,3261197.94,200412.0,4.475973e+06,420050.01,368242.80,16448.897151
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750,138109,3023206.01,135838.0,4.223108e+06,545778.28,249587.83,15248.508566


In [10]:
true_freight = pd.read_sql("SELECT SUM(Freight) AS f FROM vendor_invoice", conn)["f"].iloc[0]
print(f"Freight in summary: {vendor_sales_summary['FreightCost'].sum():,.2f} | "
      f"freight on invoices: {true_freight:,.2f}")

Freight in summary: 1,640,474.69 | freight on invoices: 1,640,474.69


## Data quality checks before cleaning

In [11]:
display(vendor_sales_summary.dtypes)
display(vendor_sales_summary.isnull().sum())

VendorNumber               int64
VendorName                   str
Brand                      int64
Description                  str
PurchasePrice            float64
ActualPrice              float64
Volume                       str
TotalPurchaseQuantity      int64
TotalPurchaseDollars     float64
TotalSalesQuantity       float64
TotalSalesDollars        float64
TotalSalesPrice          float64
TotalExciseTax           float64
FreightCost              float64
dtype: object

VendorNumber               0
VendorName                 0
Brand                      0
Description                0
PurchasePrice              0
ActualPrice                0
Volume                     0
TotalPurchaseQuantity      0
TotalPurchaseDollars       0
TotalSalesQuantity       178
TotalSalesDollars        178
TotalSalesPrice          178
TotalExciseTax           178
FreightCost                0
dtype: int64

In [12]:
# vendor names in the raw files carry trailing spaces
print(vendor_sales_summary["VendorName"].unique()[:10])

<StringArray>
['BROWN-FORMAN CORP          ',       'MARTIGNETTI COMPANIES',
 'PERNOD RICARD USA          ', 'DIAGEO NORTH AMERICA INC   ',
 'BACARDI USA INC            ', 'JIM BEAM BRANDS COMPANY    ',
 'MAJESTIC FINE WINES        ', 'ULTRA BEVERAGE COMPANY LLP ',
 'STOLI GROUP,(USA) LLC      ', 'PROXIMO SPIRITS INC.       ']
Length: 10, dtype: str


## Cleaning + new KPI columns
`clean_data()` in `src/get_vendor_summary.py`:
- `Volume` → float; missing sales (products never sold) → 0; strip spaces from names
- **GrossProfit** = sales $ − purchase $
- **ProfitMargin %** = gross profit ÷ sales $ × 100 (blank when sales are 0)
- **StockTurnover** = units sold ÷ units bought
- **SalesToPurchaseRatio** = sales $ ÷ purchase $

In [13]:
clean_df = clean_data(vendor_sales_summary)
display(clean_df.head())

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalSalesPrice,TotalExciseTax,FreightCost,GrossProfit,ProfitMargin,StockTurnover,SalesToPurchaseRatio
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750.0,145080,3811251.60,142049.0,5.101920e+06,672819.31,260999.20,19325.145490,1290667.91,25.297693,0.979108,1.338647
1,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750.0,164038,3804041.22,160247.0,4.819073e+06,561512.37,294438.66,19787.629629,1015032.27,21.062810,0.976890,1.266830
2,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750.0,187407,3418303.68,187140.0,4.538121e+06,461140.15,343854.07,17539.246213,1119816.92,24.675786,0.998575,1.327594
3,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750.0,201682,3261197.94,200412.0,4.475973e+06,420050.01,368242.80,16448.897151,1214774.94,27.139908,0.993703,1.372493
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750.0,138109,3023206.01,135838.0,4.223108e+06,545778.28,249587.83,15248.508566,1199901.61,28.412764,0.983556,1.396897


## Why save this as a table?
- The query joins ~2.4M purchase rows and ~12.8M sales rows — expensive to repeat.
- Saving the result (`python src/get_vendor_summary.py`) gives a ~10.7K-row table that the
  analysis and the Power BI dashboard can load instantly.

In [14]:
display(pd.read_sql("PRAGMA table_info(vendor_sales_summary);", conn))
conn.close()

,cid,name,type,notnull,dflt_value,pk
0,0,VendorNumber,INTEGER,0,None,0
1,1,VendorName,TEXT,0,None,0
2,2,Brand,INTEGER,0,None,0
3,3,Description,TEXT,0,None,0
4,4,PurchasePrice,REAL,0,None,0
5,5,ActualPrice,REAL,0,None,0
6,6,Volume,REAL,0,None,0
7,7,TotalPurchaseQuantity,INTEGER,0,None,0
8,8,TotalPurchaseDollars,REAL,0,None,0
9,9,TotalSalesQuantity,REAL,0,None,0
